# Krum Defence Analysis: Static Attack Scenario
## Evaluating Krum's Effectiveness Against 40% Byzantine Clients

This notebook analyzes the federated learning experiment using **Krum defence mechanism** against static label flip attacks with 40 malicious clients out of 100 total clients over 10 training rounds.

## Section 1: Load and Parse Krum Defence Log Data
Read the log file from the Krum defence experiment and extract key metrics.

In [2]:
!pip install seaborn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import re
from collections import Counter

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load the Krum defence log file
log_file = 'static_attack_krum_20260216.log'

with open(log_file, 'r') as f:
    log_content = f.read()

print("✓ Krum Defence Log Loaded Successfully")
print(f"Log size: {len(log_content)} characters")
print(f"Log lines: {len(log_content.splitlines())} lines")
print("\n" + "="*70)
print("EXPERIMENT CONFIGURATION")
print("="*70)
print("Defence Strategy: Krum")
print("Total Clients: 100")
print("Byzantine/Malicious Clients: 40 (f=40)")
print("Training Rounds: 10")
print("Dataset: MNIST (10,000 test samples)")
print("="*70)

ModuleNotFoundError: No module named 'seaborn'

## Section 2: Extract Performance Metrics
Parse centralized evaluation metrics (loss and accuracy) across all rounds.

In [ ]:
# Extract centralized evaluation metrics
centralized_evaluation = re.findall(
    r'Server Round (\d+) - CENTRALIZED EVALUATION \| Loss: ([\d.]+), Accuracy: ([\d.]+)',
    log_content
)

# Create metrics dataframe
df_metrics = pd.DataFrame(centralized_evaluation, columns=['Round', 'Loss', 'Accuracy'])
df_metrics['Round'] = df_metrics['Round'].astype(int)
df_metrics['Loss'] = df_metrics['Loss'].astype(float)
df_metrics['Accuracy'] = df_metrics['Accuracy'].astype(float)

print("✓ Performance Metrics Extracted")
print("\n" + "="*70)
print("ACCURACY & LOSS PROGRESSION")
print("="*70)
print(df_metrics.to_string(index=False))
print("="*70)

# Calculate statistics
print("\n📊 ACCURACY STATISTICS:")
print(f"   Initial (Round 0): {df_metrics['Accuracy'].iloc[0]*100:.2f}%")
print(f"   Final (Round 10):  {df_metrics['Accuracy'].iloc[10]*100:.2f}%")
print(f"   Maximum:           {df_metrics['Accuracy'].max()*100:.2f}% (Round {df_metrics['Accuracy'].idxmax()})")
print(f"   Minimum:           {df_metrics['Accuracy'].min()*100:.2f}% (Round {df_metrics['Accuracy'].idxmin()})")
print(f"   Mean:              {df_metrics['Accuracy'].mean()*100:.2f}%")
print(f"   Std Deviation:     {df_metrics['Accuracy'].std()*100:.2f}%")
print(f"   Volatility:        {(df_metrics['Accuracy'].max() - df_metrics['Accuracy'].min())*100:.2f}% range")

## Section 3: Extract Client Selection by Krum
Analyze which clients were selected by Krum in each round and their scores.

In [ ]:
# Extract accepted clients (selected by Krum)
accepted_clients = re.findall(
    r'✅ ACCEPT (client_\d+): Selected by Krum \(best score: ([\d.]+)\)',
    log_content
)

# Extract rejected clients with scores
rejected_clients = re.findall(
    r'❌ REJECT (client_\d+): Not selected by Krum \(score: ([\d.]+)\)',
    log_content
)

# Extract round aggregation info
round_aggregations = re.findall(
    r'🔄 Round (\d+): Aggregating (\d+) client updates with Krum',
    log_content
)

print("✓ Client Selection Data Extracted")
print("\n" + "="*70)
print("KRUM SELECTION SUMMARY")
print("="*70)
print(f"Total accepted decisions: {len(accepted_clients)}")
print(f"Total rejected decisions: {len(rejected_clients)}")
print(f"Selection rate: {len(accepted_clients)/(len(accepted_clients)+len(rejected_clients))*100:.2f}%")
print(f"\nAccepted per round: {len(accepted_clients)/10:.1f} clients (average)")
print(f"Rejected per round: {len(rejected_clients)/10:.1f} clients (average)")

# List accepted clients by round
print("\n" + "="*70)
print("SELECTED CLIENTS PER ROUND")
print("="*70)
for client, score in accepted_clients:
    print(f"   ✅ {client}: Krum Score = {float(score):,.2f}")

## Section 4: Analyze Krum Score Distribution
Examine the distribution of Krum scores to understand selection criteria.

In [ ]:
# Combine all scores
all_scores = [(client, float(score), 'Accepted') for client, score in accepted_clients]
all_scores.extend([(client, float(score), 'Rejected') for client, score in rejected_clients])

df_scores = pd.DataFrame(all_scores, columns=['Client', 'Krum_Score', 'Status'])

# Calculate score statistics
print("📊 KRUM SCORE DISTRIBUTION ANALYSIS")
print("="*70)
print(f"\nOverall Score Statistics:")
print(f"   Mean:   {df_scores['Krum_Score'].mean():,.2f}")
print(f"   Median: {df_scores['Krum_Score'].median():,.2f}")
print(f"   Min:    {df_scores['Krum_Score'].min():,.2f}")
print(f"   Max:    {df_scores['Krum_Score'].max():,.2f}")
print(f"   Range:  {df_scores['Krum_Score'].max() - df_scores['Krum_Score'].min():,.2f}")

print(f"\nAccepted Clients Score Statistics:")
accepted_scores = df_scores[df_scores['Status'] == 'Accepted']['Krum_Score']
print(f"   Mean:   {accepted_scores.mean():,.2f}")
print(f"   Median: {accepted_scores.median():,.2f}")
print(f"   Min:    {accepted_scores.min():,.2f}")
print(f"   Max:    {accepted_scores.max():,.2f}")

print(f"\nRejected Clients Score Statistics:")
rejected_scores = df_scores[df_scores['Status'] == 'Rejected']['Krum_Score']
print(f"   Mean:   {rejected_scores.mean():,.2f}")
print(f"   Median: {rejected_scores.median():,.2f}")
print(f"   Min:    {rejected_scores.min():,.2f}")
print(f"   Max:    {rejected_scores.max():,.2f}")

# Visualize score distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of all scores
ax1.hist(rejected_scores, bins=50, alpha=0.7, color='#E94B3C', label='Rejected', edgecolor='black')
ax1.hist(accepted_scores, bins=10, alpha=0.9, color='#06A77D', label='Accepted', edgecolor='black')
ax1.set_xlabel('Krum Score', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Krum Scores', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Box plot comparison
df_scores.boxplot(column='Krum_Score', by='Status', ax=ax2, patch_artist=True)
ax2.set_xlabel('Client Status', fontsize=12, fontweight='bold')
ax2.set_ylabel('Krum Score', fontsize=12, fontweight='bold')
ax2.set_title('Krum Score Distribution by Status', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig('krum_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Score distribution visualization created")

## Section 5: Accuracy Stability Analysis
Visualize the extreme volatility in model accuracy across training rounds.

In [ ]:
# Classify rounds as successful or failed
df_metrics['Status'] = df_metrics['Accuracy'].apply(
    lambda x: 'Attack Succeeded' if x < 0.30 else 'Defence Succeeded' if x > 0.75 else 'Uncertain'
)

# Color mapping
colors = df_metrics['Status'].map({
    'Defence Succeeded': '#06A77D',
    'Attack Succeeded': '#E94B3C',
    'Uncertain': '#FFA726'
})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Accuracy progression
ax1.plot(df_metrics['Round'], df_metrics['Accuracy'], marker='o', linewidth=2.5,
         markersize=10, color='#2E86AB', zorder=2)
ax1.scatter(df_metrics['Round'], df_metrics['Accuracy'], c=colors, s=200, 
           edgecolors='black', linewidth=2, zorder=3, alpha=0.8)

# Add threshold lines
ax1.axhline(y=0.75, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Defence Success Threshold (75%)')
ax1.axhline(y=0.30, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Attack Success Threshold (30%)')

# Shade regions
ax1.fill_between(df_metrics['Round'], 0.75, 1.0, alpha=0.1, color='green', label='Safe Zone')
ax1.fill_between(df_metrics['Round'], 0, 0.30, alpha=0.1, color='red', label='Danger Zone')

ax1.set_xlabel('Training Round', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Krum Defence: Accuracy Volatility Under Attack', fontsize=14, fontweight='bold', pad=15)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax1.set_ylim([0, 1.05])
ax1.set_xticks(range(0, 11))
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(fontsize=10, loc='center left', bbox_to_anchor=(1, 0.5))

# Add annotations for critical points
worst_round = df_metrics['Accuracy'].idxmin()
best_round = df_metrics['Accuracy'].idxmax()

ax1.annotate(f'Catastrophic Failure\n{df_metrics["Accuracy"].iloc[worst_round]*100:.2f}%',
            xy=(worst_round, df_metrics['Accuracy'].iloc[worst_round]),
            xytext=(worst_round+1, 0.15),
            fontsize=10, ha='left', color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax1.annotate(f'Peak Performance\n{df_metrics["Accuracy"].iloc[best_round]*100:.2f}%',
            xy=(best_round, df_metrics['Accuracy'].iloc[best_round]),
            xytext=(best_round-1, 0.85),
            fontsize=10, ha='right', color='green', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

# Loss progression
ax2.plot(df_metrics['Round'], df_metrics['Loss'], marker='s', linewidth=2.5,
         markersize=10, color='#A23B72', zorder=2)
ax2.scatter(df_metrics['Round'], df_metrics['Loss'], c=colors, s=200,
           edgecolors='black', linewidth=2, zorder=3, alpha=0.8)

ax2.set_xlabel('Training Round', fontsize=12, fontweight='bold')
ax2.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax2.set_title('Krum Defence: Loss Progression', fontsize=14, fontweight='bold', pad=15)
ax2.set_xticks(range(0, 11))
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('krum_accuracy_volatility.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Accuracy volatility visualization created")

## Section 6: Attack Success Rate Analysis
Calculate how often attacks succeeded in degrading model performance.

In [ ]:
# Categorize rounds (excluding initialization at round 0)
training_rounds = df_metrics[df_metrics['Round'] > 0].copy()

attack_succeeded = len(training_rounds[training_rounds['Status'] == 'Attack Succeeded'])
defence_succeeded = len(training_rounds[training_rounds['Status'] == 'Defence Succeeded'])
uncertain = len(training_rounds[training_rounds['Status'] == 'Uncertain'])
total_rounds = len(training_rounds)

print("="*70)
print("ATTACK SUCCESS RATE ANALYSIS")
print("="*70)
print(f"\nTraining Rounds Analyzed: {total_rounds} (excluding initialization)")
print(f"\nResults:")
print(f"   🛡️  Defence Succeeded: {defence_succeeded} rounds ({defence_succeeded/total_rounds*100:.1f}%)")
print(f"   ⚔️  Attack Succeeded:  {attack_succeeded} rounds ({attack_succeeded/total_rounds*100:.1f}%)")
print(f"   ❓ Uncertain:         {uncertain} rounds ({uncertain/total_rounds*100:.1f}%)")

# List rounds by status
print(f"\nDefence Success Rounds: {training_rounds[training_rounds['Status'] == 'Defence Succeeded']['Round'].tolist()}")
print(f"Attack Success Rounds:  {training_rounds[training_rounds['Status'] == 'Attack Succeeded']['Round'].tolist()}")

# Pie chart
fig, ax = plt.subplots(figsize=(10, 8))
sizes = [defence_succeeded, attack_succeeded, uncertain]
labels = ['Defence Succeeded', 'Attack Succeeded', 'Uncertain']
colors_pie = ['#06A77D', '#E94B3C', '#FFA726']
explode = (0.05, 0.05, 0.05)

wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
                                    explode=explode, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(14)

ax.set_title('Krum Defence Effectiveness: Round Outcomes Distribution', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('krum_attack_success_rate.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Attack success rate visualization created")

## Section 7: Round-by-Round Detailed Analysis
Deep dive into each round's performance and selected client.

In [ ]:
# Create detailed round analysis
print("="*80)
print("ROUND-BY-ROUND DETAILED ANALYSIS")
print("="*80)

for idx, (client, score) in enumerate(accepted_clients, start=1):
    round_data = df_metrics[df_metrics['Round'] == idx].iloc[0]
    accuracy = round_data['Accuracy']
    loss = round_data['Loss']
    status = round_data['Status']
    
    status_emoji = '✅' if status == 'Defence Succeeded' else '❌' if status == 'Attack Succeeded' else '⚠️'
    
    print(f"\n{status_emoji} ROUND {idx}:")
    print(f"   Selected Client: {client}")
    print(f"   Krum Score:      {float(score):,.2f}")
    print(f"   Accuracy:        {accuracy*100:.2f}%")
    print(f"   Loss:            {loss:.4f}")
    print(f"   Outcome:         {status}")
    
    if idx > 1:
        prev_accuracy = df_metrics[df_metrics['Round'] == idx-1]['Accuracy'].iloc[0]
        acc_change = (accuracy - prev_accuracy) * 100
        print(f"   Δ Accuracy:      {acc_change:+.2f}%")

print("\n" + "="*80)

## Section 8: Client Selection Frequency Analysis
Identify if Krum repeatedly selects the same clients.

In [ ]:
# Count client selection frequency
selected_client_names = [client for client, score in accepted_clients]
client_frequency = Counter(selected_client_names)

print("="*70)
print("CLIENT SELECTION FREQUENCY")
print("="*70)
print(f"\nTotal clients selected across all rounds: {len(selected_client_names)}")
print(f"Unique clients selected: {len(client_frequency)}")
print(f"\nSelection distribution:")

for client, count in client_frequency.most_common():
    print(f"   {client}: {count} time(s) ({count/len(accepted_clients)*100:.1f}%)")

# Visualize client selection frequency
fig, ax = plt.subplots(figsize=(12, 6))

clients = list(client_frequency.keys())
counts = list(client_frequency.values())

bars = ax.bar(clients, counts, color='#2E86AB', alpha=0.8, edgecolor='black', linewidth=1.5)

# Color bars by frequency
for bar, count in zip(bars, counts):
    if count > 1:
        bar.set_color('#E94B3C')
    else:
        bar.set_color('#06A77D')

ax.set_xlabel('Client ID', fontsize=12, fontweight='bold')
ax.set_ylabel('Selection Count', fontsize=12, fontweight='bold')
ax.set_title('Krum Client Selection Frequency (10 Rounds)', fontsize=14, fontweight='bold', pad=15)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, max(counts) + 0.5])

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('krum_client_selection_frequency.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Client selection frequency visualization created")

## Section 9: Comprehensive Comparison Dashboard
Summary dashboard showing all key metrics and findings.

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. Accuracy volatility
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df_metrics['Round'], df_metrics['Accuracy'], marker='o', linewidth=3,
         markersize=10, color='#2E86AB', label='Accuracy')
ax1.axhline(y=0.75, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax1.axhline(y=0.30, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax1.fill_between(df_metrics['Round'], 0.75, 1.0, alpha=0.1, color='green')
ax1.fill_between(df_metrics['Round'], 0, 0.30, alpha=0.1, color='red')
ax1.set_title('Accuracy Volatility Across Rounds', fontsize=13, fontweight='bold')
ax1.set_xlabel('Round', fontsize=11)
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax1.set_ylim([0, 1.05])
ax1.set_xticks(range(0, 11))
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# 2. Loss progression
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(df_metrics['Round'], df_metrics['Loss'], marker='s', linewidth=2.5,
         markersize=8, color='#A23B72')
ax2.set_title('Loss Progression', fontsize=12, fontweight='bold')
ax2.set_xlabel('Round', fontsize=10)
ax2.set_ylabel('Loss', fontsize=10)
ax2.grid(True, alpha=0.3)

# 3. Attack success pie chart
ax3 = fig.add_subplot(gs[1, 1])
sizes = [defence_succeeded, attack_succeeded, uncertain]
labels = ['Defence\nSucceeded', 'Attack\nSucceeded', 'Uncertain']
colors_pie = ['#06A77D', '#E94B3C', '#FFA726']
ax3.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
ax3.set_title('Round Outcomes', fontsize=12, fontweight='bold')

# 4. Score distribution
ax4 = fig.add_subplot(gs[1, 2])
ax4.boxplot([rejected_scores, accepted_scores], labels=['Rejected', 'Accepted'],
           patch_artist=True)
ax4.set_title('Krum Score Distribution', fontsize=12, fontweight='bold')
ax4.set_ylabel('Krum Score', fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

# 5. Client selection frequency
ax5 = fig.add_subplot(gs[2, 0])
clients = list(client_frequency.keys())
counts = list(client_frequency.values())
ax5.bar(clients, counts, color='#2E86AB', alpha=0.8, edgecolor='black')
ax5.set_title('Client Selection Frequency', fontsize=12, fontweight='bold')
ax5.set_xlabel('Client', fontsize=10)
ax5.set_ylabel('Count', fontsize=10)
ax5.tick_params(axis='x', labelsize=8, rotation=45)
ax5.grid(True, alpha=0.3, axis='y')

# 6. Accuracy change per round
ax6 = fig.add_subplot(gs[2, 1])
acc_changes = df_metrics['Accuracy'].diff() * 100
colors_bars = ['#06A77D' if c > 0 else '#E94B3C' for c in acc_changes[1:]]
ax6.bar(df_metrics['Round'][1:], acc_changes[1:], color=colors_bars, alpha=0.8, edgecolor='black')
ax6.set_title('Accuracy Change Per Round', fontsize=12, fontweight='bold')
ax6.set_xlabel('Round', fontsize=10)
ax6.set_ylabel('Δ Accuracy (%)', fontsize=10)
ax6.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax6.grid(True, alpha=0.3, axis='y')

# 7. Summary statistics table
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')
summary_text = f"""
KEY FINDINGS
━━━━━━━━━━━━━━━━━━━
Defence: Krum
Byzantine: 40/100 (40%)

ACCURACY:
• Peak:  {df_metrics['Accuracy'].max()*100:.2f}%
• Final: {df_metrics['Accuracy'].iloc[-1]*100:.2f}%
• Worst: {df_metrics['Accuracy'].min()*100:.2f}%
• Range: {(df_metrics['Accuracy'].max()-df_metrics['Accuracy'].min())*100:.2f}%

OUTCOMES:
• Defence: {defence_succeeded}/10 rounds
• Attack:  {attack_succeeded}/10 rounds
• Success: {defence_succeeded/total_rounds*100:.0f}%

SELECTION:
• Per Rd:  1 client only
• Unique:  {len(client_frequency)} clients
• Rejected: 99/100 per rd
"""

ax7.text(0.05, 0.95, summary_text, transform=ax7.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.suptitle('Krum Defence: Comprehensive Performance Analysis\n40% Byzantine Clients, Static Label Flip Attack',
            fontsize=16, fontweight='bold', y=0.995)

plt.savefig('krum_comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Comprehensive dashboard created")

## Section 10: Critical Findings and Recommendations

In [ ]:
print("="*80)
print("CRITICAL FINDINGS: KRUM DEFENCE UNDER 40% BYZANTINE ATTACK")
print("="*80)
print()

print("1. DEFENCE FAILURE - EXTREME VOLATILITY:")
print("-" * 80)
print(f"   • Accuracy swings from {df_metrics['Accuracy'].min()*100:.2f}% to {df_metrics['Accuracy'].max()*100:.2f}%")
print(f"   • Standard deviation: {df_metrics['Accuracy'].std()*100:.2f}%")
print(f"   • Catastrophic failures in {attack_succeeded} out of 10 rounds ({attack_succeeded/total_rounds*100:.0f}%)")
print(f"   • VERDICT: ❌ KRUM FAILING - Highly unstable defense")
print()

print("2. ROOT CAUSE - OVER-AGGRESSIVE SELECTION:")
print("-" * 80)
print(f"   • Krum selects ONLY 1 client per round (out of 100)")
print(f"   • 99 clients rejected every round")
print(f"   • No redundancy or averaging - single point of failure")
print(f"   • If selected client is malicious → catastrophic round")
print(f"   • VERDICT: ❌ Selection strategy too conservative")
print()

print("3. ATTACK SUCCESS PATTERN:")
print("-" * 80)
print(f"   • Attack succeeds in rounds: {training_rounds[training_rounds['Status'] == 'Attack Succeeded']['Round'].tolist()}")
print(f"   • Defence succeeds in rounds: {training_rounds[training_rounds['Status'] == 'Defence Succeeded']['Round'].tolist()}")
print(f"   • Success rate: {attack_succeeded/total_rounds*100:.0f}% of rounds compromised")
print(f"   • VERDICT: ⚔️ Attackers breach defense 40% of the time")
print()

print("4. SCORE DISTRIBUTION ISSUES:")
print("-" * 80)
print(f"   • Accepted client scores: {accepted_scores.min():,.0f} to {accepted_scores.max():,.0f}")
print(f"   • Rejected client scores: {rejected_scores.min():,.0f} to {rejected_scores.max():,.0f}")
print(f"   • Score ranges overlap significantly")
print(f"   • VERDICT: ⚠️ Score-based selection not discriminating well")
print()

print("5. THEORETICAL BREAKDOWN:")
print("-" * 80)
print(f"   • Krum assumption: f < n/2 (Byzantine clients < 50%)")
print(f"   • Current setup: f = 40, n = 100 (40% Byzantine)")
print(f"   • Operating at theoretical limit")
print(f"   • Single client selection magnifies any selection error")
print(f"   • VERDICT: ⚠️ At Krum's toleration boundary")
print()

print("="*80)
print("RECOMMENDATIONS")
print("="*80)
print()
print("IMMEDIATE FIXES:")
print("  1. Use Multi-Krum: Select top 20-30 clients instead of 1")
print("  2. Reduce Byzantine ratio: Test with 20-30% attackers")
print("  3. Add fallback: Discard rounds with accuracy drops > 50%")
print("  4. Implement momentum: Use weighted average with previous round")
print()
print("ALTERNATIVE DEFENCES TO TEST:")
print("  • Trimmed Mean (more robust to outliers)")
print("  • Median Aggregation (Byzantine-resistant)")
print("  • FoolsGold (reputation-based)")
print("  • Your Cognitive Defence mechanism")
print("  • Ensemble: Combine multiple defences")
print()
print("EXPERIMENTAL NEXT STEPS:")
print("  1. Baseline comparison: Run without any defence")
print("  2. Parameter sweep: Test f = 10, 20, 30, 40")
print("  3. Attack variants: Test adaptive vs static poisoning")
print("  4. Defence comparison: Krum vs Median vs Cognitive")
print()
print("="*80)
print("CONCLUSION: Krum alone is INSUFFICIENT against 40% Byzantine attackers.")
print("             Multi-Krum or alternative defences strongly recommended.")
print("="*80)